# Prediccion de Fallos en Servidores
## Metodo: Regresion Logistica

**Materia:** Metodos Numericos  
**Universidad:** Universidad San Francisco Xavier de Chuquisaca  

---

Este notebook implementa un modelo de **Regresion Logistica** para predecir si un servidor esta a punto de fallar, usando cuatro variables de monitoreo en tiempo real:

| Variable | Descripcion |
|---|---|
| `temperatura_cpu` | Temperatura del procesador en grados Celsius |
| `uso_cpu` | Porcentaje de uso del procesador |
| `uso_ram` | Porcentaje de uso de memoria RAM |
| `paquetes_perdidos` | Porcentaje de paquetes de red perdidos |

El modelo responde una pregunta binaria: **el servidor esta en estado Normal o hay un Fallo inminente?**


## 1. Instalacion de dependencias


In [ ]:
!pip install numpy pandas scikit-learn matplotlib ipywidgets

## 2. Carga del dataset

Se cargan 500 registros generados con distribuciones diferenciadas:
- **350 servidores normales:** temperatura ~55 C, CPU ~45%, RAM ~50%, paquetes ~2%
- **150 servidores con fallo:** temperatura ~82 C, CPU ~88%, RAM ~85%, paquetes ~18%


In [ ]:
from google.colab import files
uploaded = files.upload()

## 3. Ejecucion del modelo principal


In [ ]:
!python modelo.py

## 4. Visualizacion de resultados


In [ ]:
from IPython.display import Image
Image("resultados.png")

## 5. Fundamento matematico del metodo

### 5.1 Funcion Sigmoide

La Regresion Logistica transforma una combinacion lineal de variables en una probabilidad usando la **funcion sigmoide**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Donde $z$ es la combinacion lineal de las variables de entrada:

$$z = \beta_0 + \beta_1 \cdot x_1 + \beta_2 \cdot x_2 + \beta_3 \cdot x_3 + \beta_4 \cdot x_4$$

La sigmoide garantiza que la salida siempre este en el rango $(0, 1)$, interpretable como probabilidad.

### 5.2 Regla de decision

$$\hat{y} = \begin{cases} 1 & \text{(Fallo)} \quad \text{si } \sigma(z) \geq 0{,}5 \\ 0 & \text{(Normal)} \quad \text{si } \sigma(z) < 0{,}5 \end{cases}$$

### 5.3 Funcion de Costo: Log Loss (Entropia Cruzada Binaria)

No se utiliza el Error Cuadratico Medio (MSE) porque la composicion con la sigmoide produce una funcion no convexa con multiples minimos locales. En su lugar se usa la **Entropia Cruzada Binaria**:

$$J(\beta) = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \cdot \log(\hat{p}_i) + (1 - y_i) \cdot \log(1 - \hat{p}_i) \right]$$

Donde:
- $y_i \in \{0, 1\}$ es el valor real (0 = Normal, 1 = Fallo)
- $\hat{p}_i = \sigma(z_i)$ es la probabilidad predicha por el modelo
- $n$ es el numero total de muestras

Esta funcion es **convexa**, lo que garantiza la existencia de un unico minimo global.

### 5.4 Optimizacion

Los coeficientes $\beta$ se obtienen minimizando $J(\beta)$ mediante el algoritmo **L-BFGS** (variante del gradiente descendente para problemas convexos):

$$\beta^* = \arg\min_{\beta} \, J(\beta)$$

En cada iteracion el optimizador ajusta los coeficientes en la direccion que reduce el costo, hasta converger al minimo.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Reentrenar para tener el modelo y scaler disponibles en este entorno
df = pd.read_csv("dataset_servidores.csv")
X  = df[["temperatura_cpu", "uso_cpu", "uso_ram", "paquetes_perdidos"]]
y  = df["fallo"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Estandarizacion: lleva cada variable a media=0 y desviacion=1
# Es necesaria para que el optimizador converja correctamente
scaler    = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)

modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train_sc, y_train)

# ── Grafica 1: Funcion Sigmoide ──────────────────────────────
z        = np.linspace(-8, 8, 300)
sigmoide = 1 / (1 + np.exp(-z))

# ── Grafica 2: Log Loss para cada clase ─────────────────────
# Si y=1: costo = -log(p)     penaliza cuando p se aleja de 1
# Si y=0: costo = -log(1-p)   penaliza cuando p se aleja de 0
p        = np.linspace(0.001, 0.999, 300)
costo_y1 = -np.log(p)
costo_y0 = -np.log(1 - p)

# ── Grafica 3: Convergencia del Log Loss ─────────────────────
# Entrenamos el modelo con iteraciones crecientes para ver como baja el costo
iteraciones = list(range(1, 201, 5))
costos      = []

for it in iteraciones:
    m = LogisticRegression(max_iter=it, random_state=42, solver="lbfgs")
    m.fit(X_train_sc, y_train)
    prob_iter = np.clip(m.predict_proba(X_train_sc)[:, 1], 1e-10, 1 - 1e-10)
    # J(beta) = -(1/n) * sum( y*log(p) + (1-y)*log(1-p) )
    logloss = -np.mean(y_train * np.log(prob_iter) + (1 - y_train) * np.log(1 - prob_iter))
    costos.append(logloss)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Fundamento matematico — Regresion Logistica", fontsize=13, fontweight="bold")

# Grafica 1
ax1 = axes[0]
ax1.plot(z, sigmoide, color="#2980b9", linewidth=2.5)
ax1.axhline(y=0.5, color="gray",    linestyle="--", linewidth=1, label="Umbral = 0.5")
ax1.fill_between(z, sigmoide, 0.5, where=(sigmoide >= 0.5), alpha=0.15, color="#e74c3c", label="Zona fallo")
ax1.fill_between(z, sigmoide, 0.5, where=(sigmoide <  0.5), alpha=0.15, color="#2ecc71", label="Zona normal")
ax1.set_title("Funcion Sigmoide\ns(z) = 1 / (1 + e^(-z))", fontsize=10)
ax1.set_xlabel("z = B0 + B1*x1 + ... + Bn*xn")
ax1.set_ylabel("P(fallo)")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Grafica 2
ax2 = axes[1]
ax2.plot(p, costo_y1, color="#e74c3c", linewidth=2.5, label="y=1 (fallo real):  -log(p)")
ax2.plot(p, costo_y0, color="#2ecc71", linewidth=2.5, label="y=0 (normal real): -log(1-p)")
ax2.set_ylim(0, 5)
ax2.set_title("Funcion de Costo — Log Loss\nJ(B) = -(1/n) * S[ y*log(p) + (1-y)*log(1-p) ]", fontsize=10)
ax2.set_xlabel("Probabilidad predicha p")
ax2.set_ylabel("Costo J")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# Grafica 3
ax3 = axes[2]
ax3.plot(iteraciones, costos, color="#8e44ad", linewidth=2.5)
ax3.scatter(iteraciones[-1], costos[-1], color="#e74c3c", zorder=5,
            label=f"Log Loss final = {costos[-1]:.6f}")
ax3.set_title("Convergencia del modelo\nLog Loss vs iteraciones", fontsize=10)
ax3.set_xlabel("Iteraciones del optimizador")
ax3.set_ylabel("Log Loss J(B)")
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 55)
print("  Coeficientes aprendidos por el modelo:")
print(f"  B0 (intercepto)      : {modelo.intercept_[0]:+.4f}")
cols = ["temperatura_cpu", "uso_cpu", "uso_ram", "paquetes_perdidos"]
for col, coef in zip(cols, modelo.coef_[0]):
    print(f"  {col:<24}: {coef:+.4f}")
print("=" * 55)


## 6. Inferencia con valores propios

Modifica los 4 valores de la seccion marcada y ejecuta la celda para ver la prediccion.


In [ ]:
# ── CAMBIA ESTOS VALORES ──────────────────────
temperatura_cpu   = 75.0   # grados Celsius
uso_cpu           = 70.0   # porcentaje
uso_ram           = 65.0   # porcentaje
paquetes_perdidos =  5.0   # porcentaje
# ──────────────────────────────────────────────

nuevo    = pd.DataFrame({
    "temperatura_cpu":   [temperatura_cpu],
    "uso_cpu":           [uso_cpu],
    "uso_ram":           [uso_ram],
    "paquetes_perdidos": [paquetes_perdidos],
})
nuevo_sc = scaler.transform(nuevo)
pred     = modelo.predict(nuevo_sc)[0]
prob     = modelo.predict_proba(nuevo_sc)[0][1]

print("=" * 45)
print("  RESULTADO DEL SERVIDOR")
print("=" * 45)
print(f"  Temperatura CPU   : {temperatura_cpu} C")
print(f"  Uso CPU           : {uso_cpu} %")
print(f"  Uso RAM           : {uso_ram} %")
print(f"  Paquetes perdidos : {paquetes_perdidos} %")
print("-" * 45)
print(f"  Probabilidad de fallo : {prob:.2%}")
print(f"  Prediccion            : {'FALLO INMINENTE' if pred == 1 else 'Normal'}")
print("=" * 45)


## 7. Visualizacion interactiva

Mueve los sliders para ver como cambian en tiempo real:

- La **probabilidad de fallo** calculada por la sigmoide
- La posicion del servidor sobre la **curva sigmoide**
- El **costo Log Loss** de la prediccion actual
- La comparacion de tus variables vs el **promedio normal**


In [ ]:
import ipywidgets as widgets
from IPython.display import display

slider_temp = widgets.FloatSlider(value=75, min=30,  max=100, step=0.5,
                                  description="Temp (C):",
                                  style={"description_width": "120px"},
                                  layout=widgets.Layout(width="500px"))
slider_cpu  = widgets.FloatSlider(value=70, min=5,   max=100, step=0.5,
                                  description="Uso CPU (%):",
                                  style={"description_width": "120px"},
                                  layout=widgets.Layout(width="500px"))
slider_ram  = widgets.FloatSlider(value=65, min=10,  max=100, step=0.5,
                                  description="Uso RAM (%):",
                                  style={"description_width": "120px"},
                                  layout=widgets.Layout(width="500px"))
slider_paq  = widgets.FloatSlider(value=5,  min=0,   max=50,  step=0.5,
                                  description="Paquetes (%):",
                                  style={"description_width": "120px"},
                                  layout=widgets.Layout(width="500px"))

output_int = widgets.Output()

def actualizar(temp, cpu, ram, paq):
    nuevo = pd.DataFrame({
        "temperatura_cpu":   [temp],
        "uso_cpu":           [cpu],
        "uso_ram":           [ram],
        "paquetes_perdidos": [paq],
    })
    nuevo_sc = scaler.transform(nuevo)
    prob     = modelo.predict_proba(nuevo_sc)[0][1]
    pred     = modelo.predict(nuevo_sc)[0]
    z_val    = modelo.decision_function(nuevo_sc)[0]

    color = "#e74c3c" if prob >= 0.5 else "#2ecc71"

    with output_int:
        output_int.clear_output(wait=True)

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle("Regresion Logistica — Visualizacion interactiva", fontsize=14, fontweight="bold")

        # Grafica 1: barra de probabilidad
        ax1 = axes[0][0]
        ax1.barh(["P(fallo)"], [prob],     color=color,     height=0.4)
        ax1.barh(["P(fallo)"], [1 - prob], color="#ecf0f1", height=0.4, left=prob)
        ax1.axvline(x=0.5, color="gray", linestyle="--", linewidth=1.5, label="Umbral 0.5")
        ax1.set_xlim(0, 1)
        ax1.set_xlabel("Probabilidad")
        ax1.set_title(f"Probabilidad de fallo: {prob:.2%}")
        estado = "FALLO INMINENTE" if pred == 1 else "Normal"
        ax1.text(0.5, -0.45, estado, ha="center", transform=ax1.transAxes,
                 fontsize=13, fontweight="bold", color=color)
        ax1.legend(fontsize=9)

        # Grafica 2: comparacion de variables
        ax2 = axes[0][1]
        variables = ["Temp", "CPU", "RAM", "Paquetes"]
        valores   = [temp/100, cpu/100, ram/100, paq/50]
        normales  = [0.55,     0.45,    0.50,    0.04]
        x = np.arange(len(variables))
        w = 0.35
        ax2.bar(x - w/2, normales, w, label="Promedio normal", color="#3498db", alpha=0.7)
        ax2.bar(x + w/2, valores,  w, label="Tu servidor",     color=color,    alpha=0.9)
        ax2.set_xticks(x)
        ax2.set_xticklabels(variables)
        ax2.set_ylim(0, 1.15)
        ax2.set_ylabel("Valor normalizado (0-1)")
        ax2.set_title("Tus variables vs promedio normal")
        ax2.axhline(y=0.7, color="orange", linestyle=":", linewidth=1.5, label="Zona de riesgo")
        ax2.legend(fontsize=9)

        # Grafica 3: sigmoide con punto actual
        ax3 = axes[1][0]
        z_rango  = np.linspace(-8, 8, 300)
        sig      = 1 / (1 + np.exp(-z_rango))
        ax3.plot(z_rango, sig, color="#2980b9", linewidth=2.5, label="Sigmoide")
        ax3.axhline(y=0.5, color="gray", linestyle="--", linewidth=1, label="Umbral 0.5")
        ax3.fill_between(z_rango, sig, 0.5, where=(sig >= 0.5), alpha=0.12, color="#e74c3c")
        ax3.fill_between(z_rango, sig, 0.5, where=(sig <  0.5), alpha=0.12, color="#2ecc71")
        z_clamp = np.clip(z_val, -8, 8)
        ax3.scatter([z_clamp], [prob], color=color, s=120, zorder=5,
                    label=f"Tu servidor (z={z_val:.2f})")
        ax3.axvline(x=z_clamp, color=color, linestyle=":", linewidth=1.2)
        ax3.axhline(y=prob,    color=color, linestyle=":", linewidth=1.2)
        ax3.set_xlabel("z (valor de riesgo calculado)")
        ax3.set_ylabel("Probabilidad")
        ax3.set_title("Sigmoide: s(z) = 1 / (1 + e^(-z))")
        ax3.legend(fontsize=9)
        ax3.grid(True, alpha=0.3)

        # Grafica 4: Log Loss con costo actual
        ax4 = axes[1][1]
        p_rango  = np.linspace(0.001, 0.999, 300)
        ax4.plot(p_rango, -np.log(p_rango),       color="#e74c3c", linewidth=2, label="y=1 (fallo real)")
        ax4.plot(p_rango, -np.log(1 - p_rango),   color="#2ecc71", linewidth=2, label="y=0 (normal real)")
        ax4.set_ylim(0, 5)
        costo_actual = -np.log(max(prob, 1e-10)) if pred == 1 else -np.log(max(1 - prob, 1e-10))
        ax4.scatter([prob], [costo_actual], color=color, s=120, zorder=5,
                    label=f"Costo actual: {costo_actual:.3f}")
        ax4.axvline(x=prob, color=color, linestyle=":", linewidth=1.2)
        ax4.set_xlabel("Probabilidad predicha")
        ax4.set_ylabel("Costo Log Loss")
        ax4.set_title("Log Loss: J = -(1/n)*S[y*log(p)+(1-y)*log(1-p)]")
        ax4.legend(fontsize=9)
        ax4.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        print(f"  z calculado    : {z_val:.4f}")
        print(f"  P(fallo)       : {prob:.4f}  ({prob:.2%})")
        print(f"  Costo Log Loss : {costo_actual:.4f}")
        print(f"  Prediccion     : {estado}")

ui = widgets.VBox([slider_temp, slider_cpu, slider_ram, slider_paq, output_int])
display(ui)

widgets.interactive_output(actualizar, {
    "temp": slider_temp,
    "cpu":  slider_cpu,
    "ram":  slider_ram,
    "paq":  slider_paq,
})


## 8. Conclusion

El modelo de Regresion Logistica implementado demuestra que es posible predecir fallos en servidores con alta precision a partir de variables de monitoreo.

**Aspectos matematicos clave verificados:**

1. **Funcion sigmoide** — convierte cualquier valor real $z$ en una probabilidad entre 0 y 1
2. **Log Loss** — funcion de costo convexa que garantiza un unico minimo global
3. **Convergencia** — el optimizador reduce el costo iteracion a iteracion hasta estabilizarse
4. **Coeficientes $\beta$** — los pesos aprendidos confirman que CPU y temperatura son las variables mas determinantes

**Resultados en el conjunto de prueba:**
- Precision: 1,00
- Recall: 1,00  
- F1-score: 1,00
- AUC-ROC: 1,0000
